# Chain and group iterables

Five tools, each in a different job: `chain` joins iterables end-to-end; `groupby` collects runs of equal values; `zip` and `zip_longest` run iterables in parallel; `product` and friends generate combinations. None of them materialise their inputs, which is why they compose so well.

## `chain` — one after another

`itertools.chain(*iterables)` yields all values from its first argument, then all from the second, and so on. No intermediate list; the arguments can be any iterables, including generators or the output of other `itertools` calls.

In [ ]:
from itertools import chain

a = [1, 2, 3]
b = range(10, 13)
c = (x * x for x in [4, 5])

for v in chain(a, b, c):
    print(v, end=' ')
print()

### `chain.from_iterable` — flatten one level

When you have an iterable *of* iterables — a list of lists, a generator of tuples — use `chain.from_iterable`. It's identical to `chain(*nested)` but doesn't force the outer iterable into memory.

In [ ]:
from itertools import chain

rows = [[1, 2, 3], [4, 5], [6]]
print(list(chain.from_iterable(rows)))

# Handy for flattening one level of nesting in a pipeline:
groups = ({'a': 1, 'b': 2}, {'c': 3})
print(list(chain.from_iterable(d.items() for d in groups)))

It does *not* flatten recursively. For deep flatten you'd write a small generator (see notebook 2).

## `groupby` — runs of equal values

`groupby(iterable, key=...)` yields `(key, subiterator)` pairs. The catch — it only groups **adjacent** equal values. Like the Unix `uniq` command, not SQL `GROUP BY`.

If your data is already sorted by the key, `groupby` is a single pass. If it isn't, sort first.

In [ ]:
from itertools import groupby

events = [
    ('2024-01', 'login'),
    ('2024-01', 'click'),
    ('2024-02', 'login'),
    ('2024-02', 'login'),
    ('2024-03', 'click'),
]

for month, items in groupby(events, key=lambda e: e[0]):
    print(month, [action for _, action in items])

### Sort-then-group for unsorted input

In [ ]:
from itertools import groupby

data = [('a', 1), ('b', 2), ('a', 3), ('c', 4), ('b', 5)]

data_sorted = sorted(data, key=lambda p: p[0])
for key, group in groupby(data_sorted, key=lambda p: p[0]):
    total = sum(v for _, v in group)
    print(f'{key}: {total}')

### Consuming the sub-iterator

The sub-iterator is valid only while you're on that iteration of the outer loop. Once the outer loop advances, the previous sub-iterator is exhausted. This trips people up.

In [ ]:
from itertools import groupby

items = [('a', 1), ('a', 2), ('b', 3), ('b', 4)]

# Bug: save the iterators, try to use them later
saved = []
for key, group in groupby(items, key=lambda p: p[0]):
    saved.append((key, group))

for key, group in saved:
    print(key, list(group))     # all empty!

In [ ]:
# Fix: materialise into a list inside the loop
materialised = []
for key, group in groupby(items, key=lambda p: p[0]):
    materialised.append((key, list(group)))    # list NOW

for key, group in materialised:
    print(key, group)

### Counting per group

Common enough to be worth highlighting: when you only want counts, reach for `collections.Counter` instead of `groupby` — no sorting required, no adjacency requirement.

In [ ]:
from collections import Counter

data = ['a', 'b', 'a', 'c', 'b', 'a']
print(Counter(data))

## `zip` — parallel iteration

Built-in `zip` yields tuples from each iterable in lockstep, stopping at the shortest. It's an iterator; use `list(...)` to see the values.

In [ ]:
names  = ['Ada', 'Grace', 'Linus']
scores = [95, 88, 72]

for name, score in zip(names, scores):
    print(f'{name}: {score}')

### `zip(..., strict=True)` — fail loudly on length mismatch

Since Python 3.10. Stops silent truncation from hiding bugs.

In [ ]:
try:
    list(zip([1, 2, 3], [10, 20], strict=True))
except ValueError as e:
    print(f'caught: {e}')

### `zip_longest` — pad instead of stop

From `itertools`. When iterables are of different length, pad missing values with `fillvalue` (default `None`). Good for aligning ragged data.

In [ ]:
from itertools import zip_longest

a = [1, 2, 3, 4]
b = ['x', 'y']
print(list(zip_longest(a, b, fillvalue='?')))

### Transposing rows to columns

`zip` does matrix transpose for free: `list(zip(*rows))`.

In [ ]:
rows = [
    (1, 2, 3),
    (4, 5, 6),
    (7, 8, 9),
]

cols = list(zip(*rows))
print(cols)

## `product` — Cartesian product (cross-join)

`itertools.product(a, b, ...)` yields every combination of one value from each iterable. For two iterables it's what nested `for` loops produce; for n iterables it's a cross-join.

In [ ]:
from itertools import product

sizes = ['S', 'M', 'L']
colours = ['red', 'blue']

for s, c in product(sizes, colours):
    print(f'{s}-{c}')

In [ ]:
# Equivalent to nested loops — but lazy
from itertools import product
print(len(list(product(range(3), range(4), range(5)))))   # 3*4*5 = 60

`product(iter, repeat=n)` is `product(iter, iter, ..., iter)` (n copies). Useful for "all n-tuples from this alphabet".

In [ ]:
from itertools import product

print(list(product('01', repeat=3)))

## `combinations` and `permutations` — the other combinatorics

In [ ]:
from itertools import combinations, permutations

# combinations: order doesn't matter, no repeats
print(list(combinations('abcd', 2)))

# permutations: order matters, no repeats
print(list(permutations('abc', 2)))

## A worked example

Given two teams' scores, compute per-team totals, the combined ordered list, and the pairs with the same rank.

In [ ]:
from itertools import chain, groupby

team_a = [('Alice', 12), ('Bob', 18), ('Carol', 9)]
team_b = [('Dan', 12), ('Eve', 20), ('Fern', 18)]

# 1. combined, sorted by score desc
combined = sorted(chain(team_a, team_b), key=lambda p: -p[1])
for name, score in combined:
    print(f'{name:>6}: {score}')

In [ ]:
# 2. group by identical score
for score, players in groupby(combined, key=lambda p: p[1]):
    players = [p[0] for p in players]
    if len(players) > 1:
        print(f'tied at {score}: {players}')

In [ ]:
# 3. per-team total with zip
from itertools import zip_longest

a_sorted = sorted(team_a, key=lambda p: -p[1])
b_sorted = sorted(team_b, key=lambda p: -p[1])

for a, b in zip_longest(a_sorted, b_sorted, fillvalue=('-', 0)):
    print(f'{a[0]:>6}({a[1]:>2}) vs {b[0]:>6}({b[1]:>2})')

## Summary

- `chain`: concatenate iterables without materialising.
- `chain.from_iterable`: flatten one level of nested iterables.
- `groupby`: runs of *adjacent* equal values — sort first if needed, and materialise groups inside the loop.
- `zip` / `zip_longest`: parallel iteration. Use `strict=True` to catch length bugs.
- `product` / `combinations` / `permutations`: combinatorial generators.

These are generators themselves, so they slot into any pipeline: `sum(x+y for x, y in zip(a, b))`, `sorted(chain(...))`, and so on.